# Inference experiments

Parse a sample email, build a combined GLiNER2 schema from the taxonomy YAMLs, and inspect the results.

Run the model cell once; edit the taxonomy list or YAMLs and re-run from **Schema** onwards.

In [ ]:
import json
import sys
from pathlib import Path
from types import SimpleNamespace

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from incolor.extraction.model.loading import load_model
from incolor.parsing import EmailParser
from incolor.extraction.taxonomy.loader import TaskLoader
from incolor.extraction.taxonomy.schema import SchemaBuilder

\\bluecrest.local\cfs\Home\angel.adeoye\Projects\dev\financial_research\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
CANDIDATES = ROOT / "research_emails_20260930_14d" / "candidates_emails_nonempty_text_html.jsonl"
TAXONOMY_DIR = ROOT / "config"
SAMPLE_LINE = 10
MODEL_SOURCE = "fastino/gliner2.5-base-v1"

# Set to a string to skip the email sample and test arbitrary text.
CUSTOM_TEXT = (
    "The start of the week has brought more dollar strength. Firmer oil prices, following increased "
    "military activity in the Gulf, have helped keep the dollar on solid footing. Today’s calendar is "
    "unlikely to be particularly impactful, with the NFIB survey and home sales figures the main "
    "highlights. "
    "That said, several technical indicators point to an increasingly stretched USD rally. Our shortterm "
    "valuation model shows the dollar overvalued against every G10 currency except GBP, "
    "where it’s at fair value. On an equally weighted basis, USD is 1.1% overvalued versus G10. EUR, "
    "SEK and NZD stand out as particularly cheap, undershooting fair value by 1.5-2.5% and sitting "
    "beyond the 1.5 standard deviation threshold."
)

TAXONOMY_FILES = [
    # "taxonomies/extraction/entity.yaml",
    # "extraction/geography.yaml",
    # "extraction/financial_instrument.yaml",
    # "extraction/market_reference.yaml",
    # "extraction/organization.yaml",
    # "extraction/person.yaml",
    # "taxonomies/classification/asset_class.yaml",
    # "taxonomies/classification/market_coverage.yaml",
    # "taxonomies/classification/theme.yaml",
    "experiments/claims/claims_v2_exclusive_fields.yaml"
]

## Sample

In [3]:
def load_record(line_number: int) -> dict:
    with CANDIDATES.open(encoding="utf-8-sig") as stream:
        for number, line in enumerate(stream, 1):
            if number == line_number:
                return json.loads(line)
    raise RuntimeError(f"Line {line_number} not found in {CANDIDATES}")


def parse_record(record: dict):
    message = SimpleNamespace(
        HTMLBody=record["html_body"],
        Body="",
        Subject=record["subject"],
        SenderEmailAddress=record["sender"],
        ReceivedTime=record["received"],
    )
    return EmailParser().parse(message, attachments=())


if CUSTOM_TEXT is None:
    parsed = parse_record(load_record(SAMPLE_LINE))
    text = parsed.as_text
else:
    text = CUSTOM_TEXT

print(text)

The start of the week has brought more dollar strength. Firmer oil prices, following increased military activity in the Gulf, have helped keep the dollar on solid footing. Today’s calendar is unlikely to be particularly impactful, with the NFIB survey and home sales figures the main highlights. That said, several technical indicators point to an increasingly stretched USD rally. Our shortterm valuation model shows the dollar overvalued against every G10 currency except GBP, where it’s at fair value. On an equally weighted basis, USD is 1.1% overvalued versus G10. EUR, SEK and NZD stand out as particularly cheap, undershooting fair value by 1.5-2.5% and sitting beyond the 1.5 standard deviation threshold.


## Model (run once)

In [4]:
model = load_model(MODEL_SOURCE)

You are using a model of type gliner2 to instantiate a model of type extractor. This is not supported for all configurations of models and can yield errors.
\\bluecrest.local\cfs\Home\angel.adeoye\Projects\dev\financial_research\.venv\Lib\site-packages\gliner2\models\boundary\model.py:2212: UserWarning: Checkpoint uses legacy list-valued extra_special_tokens metadata; loading with compatibility normalization.
  tokenizer = load_extractor_tokenizer(tokenizer_source)


## Schema

Re-run after editing any taxonomy YAML.

In [12]:
tasks = [TaskLoader.load(TAXONOMY_DIR / rel) for rel in TAXONOMY_FILES]
schema = SchemaBuilder(model).build(tasks)
print(f"{len(tasks)} tasks: {[t.id for t in tasks]}")

1 tasks: ['exp_claims_v2_exclusive_fields']


## Extract

In [13]:
def flatten(results: dict):
    """Yield (task, value, confidence) rows from a formatted extract() result."""
    for task, value in results.items():
        if value is None:
            continue
        if isinstance(value, list):
            for item in value:
                if isinstance(item, dict):
                    yield task, item.get("text", item.get("label", "")), item.get("confidence", "")
                else:
                    yield task, item, ""
        elif isinstance(value, dict):
            yield task, value.get("label", value.get("text", "")), value.get("confidence", "")
        else:
            yield task, value, ""


results = model.extract_long(
    text, 
    schema, 
    chunk_size=512,
    chunk_overlap=96,
    include_confidence=True,
    overlap_policy = "flat"
)
rows = list(flatten(results))

width = max((len(task) for task, _, _ in rows), default=4)
for task, value, confidence in rows:
    conf = f"{confidence:.3f}" if isinstance(confidence, float) else confidence
    print(f"{task:<{width}}  {conf!s:<6}  {value}")

claim  0.507   


## Raw results

In [14]:
results

{'claim': [{'view': None,
   'forecast': None,
   'trade_idea': None,
   'valuation': {'text': 'overvalued', 'confidence': 0.5363330245018005},
   'positioning': None,
   'data_point': None,
   'confidence': 0.5068657398223877}]}

2. Strict definitions of the five claim types
Claim type	Strict definition	Required distinguishing feature
TRADE_IDEA	An explicit recommendation or proposed strategy to establish, modify, hedge or close a market exposure	Actionable trading direction or strategy
VIEW	An analyst's substantive assessment of a financially relevant subject's current condition, drivers, risks or investment outlook	Interpretive judgment or conviction
FORECAST	An analyst's expressed expectation of a future financial, economic or policy outcome	Predicted future outcome
VALUATION	An assessment of an asset's or instrument's value relative to an explicitly stated or identifiable valuation benchmark	Fair value, mispricing or relative cheapness/richness
POSITIONING	A statement describing actual or estimated exposures, holdings, allocations or open positions of identifiable market participants	Existing market exposure or allocation
These are conceptual definitions. For reliable model training, we also need positive inclusion criteria and explicit exclusion rules for each class.

In [51]:

print("Architecture:", model.config.architecture)
print("Max length:", model.config.max_len)

print(
    "Parameters:",
    f"{sum(p.numel() for p in model.parameters()):,}",
)

print("Configuration:")
print(model.config)


Architecture: boundary
Max length: 4096
Parameters: 193,581,591
Configuration:
ExtractorConfig {
  "architecture": "boundary",
  "architecture_version": 1,
  "architectures": [
    "Gliner2ForSchemaExtraction"
  ],
  "attn_implementation": "sdpa",
  "boundary_head": {
    "abstention_loss_weight": 0.2,
    "abstention_threshold": 0.5,
    "adaptive_threshold": false,
    "bidirectional_proposals": true,
    "boundary_attention_heads": 4,
    "boundary_attention_layers": 2,
    "boundary_attention_window": 128,
    "boundary_dim": 128,
    "boundary_ffn_multiplier": 2.0,
    "boundary_focal_clip": 0.05,
    "boundary_focal_gamma_negative": 2.0,
    "boundary_focal_gamma_positive": 0.0,
    "boundary_marginal_loss": "asymmetric_focal",
    "boundary_negative_weight": 0.5,
    "boundary_refinement_layers": 1,
    "boundary_top_k_alpha": 0.08,
    "boundary_top_k_bucket": 8,
    "boundary_top_k_max": 128,
    "candidate_attention_heads": 4,
    "candidate_attention_layers": 0,
    "candida